<a href="https://colab.research.google.com/github/jonash-chataut/GenAI/blob/main/tool_calling_in_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import userdata
HUGGINGFACEHUB_ACCESS_TOKEN=userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

In [ ]:
!pip install -q langchain_huggingface

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm_hf=HuggingFaceEndpoint(
    repo_id="MiniMaxAI/MiniMax-M2.5",
    task="text-generation",
    huggingfacehub_api_token=HUGGINGFACEHUB_ACCESS_TOKEN
)
llm=ChatHuggingFace(llm=llm_hf)

In [ ]:
!pip install -q langchain-openai langchain-core requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 5.8 MB/s eta 0:00:00


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [ ]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [ ]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [ ]:
multiply.name

'multiply'

In [ ]:
multiply.description

'Given 2 numbers a and b this tool returns their product'

In [ ]:
multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [ ]:
# tool binding

In [ ]:
# llm = ChatOpenAI()

In [ ]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 38, 'total_tokens': 73}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d01ce-cbe3-7743-9c8a-bffb17589e32-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 38, 'output_tokens': 35, 'total_tokens': 73})

In [ ]:
llm_with_tools = llm.bind_tools([multiply])

In [ ]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="Hi there! I'm doing well, thank you for asking! How can I help you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 207, 'total_tokens': 259}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d01ce-dcb6-7261-90a1-0c5526a1bb64-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 207, 'output_tokens': 52, 'total_tokens': 259})

In [ ]:
query = HumanMessage('can you multiply 3 with 1000')

In [ ]:
messages = [query]

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [ ]:
result = llm_with_tools.invoke(messages)

In [ ]:
messages.append(result)

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-a813c6d5e8fb3156', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 212, 'total_tokens': 342}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d01ce-eedb-74d0-a4c3-266f1fbc23ca-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-a813c6d5e8fb3156', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 212, 'output_tokens': 130, 'total_tokens': 342})]

In [ ]:
tool_result = multiply.invoke(result.tool_calls[0])

In [ ]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-a813c6d5e8fb3156')

In [ ]:
messages.append(tool_result)

In [ ]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"a": 3, "b": 1000}', 'name': 'multiply', 'description': None}, 'id': 'chatcmpl-tool-a813c6d5e8fb3156', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 130, 'prompt_tokens': 212, 'total_tokens': 342}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d01ce-eedb-74d0-a4c3-266f1fbc23ca-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'chatcmpl-tool-a813c6d5e8fb3156', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 212, 'output_tokens': 130, 'total_tokens': 342}),
 ToolMessage(content='3000', name='multiply', tool_call_id='chatcmpl-tool-a813c6d5e8fb3156')]

In [ ]:
llm_with_tools.invoke(messages).content

'3 multiplied by 1000 equals **3000**.'

In [ ]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/04aad474c627dd4ae6bca6a5/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  This tool converts a currency value using a conversion rate obtained from another tool.
  The conversion_rate is automatically provided from previous tool output.
  """

  return base_currency_value * conversion_rate


In [ ]:
convert.args

{'base_currency_value': {'title': 'Base Currency Value', 'type': 'integer'}}

In [ ]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'NPR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1773792001,
 'time_last_update_utc': 'Wed, 18 Mar 2026 00:00:01 +0000',
 'time_next_update_unix': 1773878401,
 'time_next_update_utc': 'Thu, 19 Mar 2026 00:00:01 +0000',
 'base_code': 'USD',
 'target_code': 'NPR',
 'conversion_rate': 147.9598}

In [ ]:
convert.invoke({'base_currency_value':10, 'conversion_rate':147.9598})

1479.598

In [ ]:
# tool binding
llm=ChatHuggingFace(llm=llm_hf)

In [ ]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [ ]:
messages = [HumanMessage('What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr')]

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr', additional_kwargs={}, response_metadata={})]

In [ ]:
ai_message = llm_with_tools.invoke(messages)

In [ ]:
messages.append(ai_message)

In [ ]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_currency': 'USD', 'target_currency': 'NPR'},
  'id': 'chatcmpl-tool-8abd691303fa9bf4',
  'type': 'tool_call'}]

In [ ]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [ ]:
messages

[HumanMessage(content='What is the conversion factor between USD and NPR, and based on that convert 100 usd to npr', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"base_currency": "USD", "target_currency": "NPR"}', 'name': 'get_conversion_factor', 'description': None}, 'id': 'chatcmpl-tool-8abd691303fa9bf4', 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 353, 'prompt_tokens': 311, 'total_tokens': 664}, 'model_name': 'MiniMaxAI/MiniMax-M2.5', 'system_fingerprint': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d0218-a045-71c3-979f-f0b6041bb451-0', tool_calls=[{'name': 'get_conversion_factor', 'args': {'base_currency': 'USD', 'target_currency': 'NPR'}, 'id': 'chatcmpl-tool-8abd691303fa9bf4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 311, 'output_tokens': 353, 'total_tokens': 664}),
 ToolMessage(content='{"result":

In [ ]:
llm_with_tools.invoke(messages).content

'The conversion factor between USD and NPR is **1 USD = 147.9598 NPR**.\n\nBased on that conversion factor, **100 USD** would be equal to **14,795.98 NPR**.'

In [ ]:
from langchain.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)

In [ ]:
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})



> Entering new AgentExecutor chain...
I'm here and ready to help! How can I assist you today?

> Finished chain.
